In [1]:
# 0. importing related libraries

import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer


In [2]:
# 1A. loading data
df = pd.read_csv("Dataset_campus_wifi_Raw.csv")
df

,Record_ID,Timestamp_Hour,Day_of_Week,Location,Num_Connected_Devices,Bandwidth_Usage_Mbps,AP_Load_Percent,Network_Latency_ms,Packet_Loss_Rate_Percent,Signal_Strength_dBm,Retransmission_Rate_Percent,Throughput_Mbps,Active_Sessions,Channel_Utilization_Percent,Congestion_Level
0,5077,15,Friday,Cafeteria,136.904535,159.14,36.80,39.7,1.72,-55.6,2.82,137.27175603882773,210.0,28.86,Low
1,4532,9,Monday,Library,58.567281,64.59,26.38,34.9,1.22,-47.2,2.06,52.81640466772333,80.0,22.63,Low
2,1498,20,Wednesday,Lecture_Hall_1,5.000000,5.00,2.73,14.9,0.15,-40.7,0.29,4.606721684454397,30.0,5.91,Low
3,5454,21,Tuesday,Hostel_B,59.546069,43.90,19.00,NaN,1.04,-43.2,NaN,39.06242858329659,88.0,14.41,Low
4,5512,7,Thursday,Lab_Block_B,50.213885,46.81,13.75,31.1,0.62,-48.3,1.70,43.310415576262486,40.0,7.15,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5570,2769,6,Thursday,Admin_Block,20.700667,5.00,6.24,22.8,0.53,-42.9,0.83,4.887467543307316,30.0,9.34,Low
5571,2738,19,Tuesday,Admin_Block,41.242511,50.04,8.45,18.4,1.66,-44.6,2.61,39.44950184149084,45.0,11.62,Low
5572,4241,22,Friday,Lecture_Hall_1,6.874638,5.00,2.14,20.6,0.26,-42.6,0.56,4.883837562481263,5.0,0.00,Low
5573,6306,12,Tuesday,Lecture_Hall_2,161.778438,173.69,65.59,63.5,2.71,-57.5,4.10,148.4316333526678,220.0,64.33,Low


In [3]:
# 1B. Initial Inspection
df.shape
df.head()
df.info()
df.describe()      
df.isnull().sum()  
df.duplicated().sum()  
df['Congestion_Level'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5575 entries, 0 to 5574
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Record_ID                    5575 non-null   int64  
 1   Timestamp_Hour               5575 non-null   int64  
 2   Day_of_Week                  5575 non-null   object 
 3   Location                     5575 non-null   object 
 4   Num_Connected_Devices        5575 non-null   float64
 5   Bandwidth_Usage_Mbps         5185 non-null   float64
 6   AP_Load_Percent              5165 non-null   float64
 7   Network_Latency_ms           5254 non-null   float64
 8   Packet_Loss_Rate_Percent     5132 non-null   float64
 9   Signal_Strength_dBm          5096 non-null   float64
 10  Retransmission_Rate_Percent  5321 non-null   float64
 11  Throughput_Mbps              5575 non-null   object 
 12  Active_Sessions              5270 non-null   float64
 13  Channel_Utilizatio

Congestion_Level
Low       5129
Medium     280
High        86
medium      21
MEDIUM      21
high        19
LOW         19
Name: count, dtype: int64

In [4]:
# 2. Data Cleaning (Preprocessing Part 1)

In [5]:
# 2A. Fix label inconsistencies (target column)

df['Congestion_Level'] = df['Congestion_Level'].str.strip().str.title()             
df

,Record_ID,Timestamp_Hour,Day_of_Week,Location,Num_Connected_Devices,Bandwidth_Usage_Mbps,AP_Load_Percent,Network_Latency_ms,Packet_Loss_Rate_Percent,Signal_Strength_dBm,Retransmission_Rate_Percent,Throughput_Mbps,Active_Sessions,Channel_Utilization_Percent,Congestion_Level
0,5077,15,Friday,Cafeteria,136.904535,159.14,36.80,39.7,1.72,-55.6,2.82,137.27175603882773,210.0,28.86,Low
1,4532,9,Monday,Library,58.567281,64.59,26.38,34.9,1.22,-47.2,2.06,52.81640466772333,80.0,22.63,Low
2,1498,20,Wednesday,Lecture_Hall_1,5.000000,5.00,2.73,14.9,0.15,-40.7,0.29,4.606721684454397,30.0,5.91,Low
3,5454,21,Tuesday,Hostel_B,59.546069,43.90,19.00,NaN,1.04,-43.2,NaN,39.06242858329659,88.0,14.41,Low
4,5512,7,Thursday,Lab_Block_B,50.213885,46.81,13.75,31.1,0.62,-48.3,1.70,43.310415576262486,40.0,7.15,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5570,2769,6,Thursday,Admin_Block,20.700667,5.00,6.24,22.8,0.53,-42.9,0.83,4.887467543307316,30.0,9.34,Low
5571,2738,19,Tuesday,Admin_Block,41.242511,50.04,8.45,18.4,1.66,-44.6,2.61,39.44950184149084,45.0,11.62,Low
5572,4241,22,Friday,Lecture_Hall_1,6.874638,5.00,2.14,20.6,0.26,-42.6,0.56,4.883837562481263,5.0,0.00,Low
5573,6306,12,Tuesday,Lecture_Hall_2,161.778438,173.69,65.59,63.5,2.71,-57.5,4.10,148.4316333526678,220.0,64.33,Low


In [6]:
# 2B. Fix Day_of_Week

day_map = {'Mon':'Monday','Tue':'Tuesday','Wed':'Wednesday',
           'Thu':'Thursday','Fri':'Friday','Sat':'Saturday','Sun':'Sunday'}
df['Day_of_Week'] = df['Day_of_Week'].str.strip().str.title()
df['Day_of_Week'] = df['Day_of_Week'].replace(day_map)                               
df

,Record_ID,Timestamp_Hour,Day_of_Week,Location,Num_Connected_Devices,Bandwidth_Usage_Mbps,AP_Load_Percent,Network_Latency_ms,Packet_Loss_Rate_Percent,Signal_Strength_dBm,Retransmission_Rate_Percent,Throughput_Mbps,Active_Sessions,Channel_Utilization_Percent,Congestion_Level
0,5077,15,Friday,Cafeteria,136.904535,159.14,36.80,39.7,1.72,-55.6,2.82,137.27175603882773,210.0,28.86,Low
1,4532,9,Monday,Library,58.567281,64.59,26.38,34.9,1.22,-47.2,2.06,52.81640466772333,80.0,22.63,Low
2,1498,20,Wednesday,Lecture_Hall_1,5.000000,5.00,2.73,14.9,0.15,-40.7,0.29,4.606721684454397,30.0,5.91,Low
3,5454,21,Tuesday,Hostel_B,59.546069,43.90,19.00,NaN,1.04,-43.2,NaN,39.06242858329659,88.0,14.41,Low
4,5512,7,Thursday,Lab_Block_B,50.213885,46.81,13.75,31.1,0.62,-48.3,1.70,43.310415576262486,40.0,7.15,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5570,2769,6,Thursday,Admin_Block,20.700667,5.00,6.24,22.8,0.53,-42.9,0.83,4.887467543307316,30.0,9.34,Low
5571,2738,19,Tuesday,Admin_Block,41.242511,50.04,8.45,18.4,1.66,-44.6,2.61,39.44950184149084,45.0,11.62,Low
5572,4241,22,Friday,Lecture_Hall_1,6.874638,5.00,2.14,20.6,0.26,-42.6,0.56,4.883837562481263,5.0,0.00,Low
5573,6306,12,Tuesday,Lecture_Hall_2,161.778438,173.69,65.59,63.5,2.71,-57.5,4.10,148.4316333526678,220.0,64.33,Low


In [7]:
# 2C. Fix Throughput_Mbps — strip string " Mbps" suffix

df['Throughput_Mbps'] = df['Throughput_Mbps'].astype(str).str.replace(' Mbps','', regex=False).astype(float)      
df

,Record_ID,Timestamp_Hour,Day_of_Week,Location,Num_Connected_Devices,Bandwidth_Usage_Mbps,AP_Load_Percent,Network_Latency_ms,Packet_Loss_Rate_Percent,Signal_Strength_dBm,Retransmission_Rate_Percent,Throughput_Mbps,Active_Sessions,Channel_Utilization_Percent,Congestion_Level
0,5077,15,Friday,Cafeteria,136.904535,159.14,36.80,39.7,1.72,-55.6,2.82,137.271756,210.0,28.86,Low
1,4532,9,Monday,Library,58.567281,64.59,26.38,34.9,1.22,-47.2,2.06,52.816405,80.0,22.63,Low
2,1498,20,Wednesday,Lecture_Hall_1,5.000000,5.00,2.73,14.9,0.15,-40.7,0.29,4.606722,30.0,5.91,Low
3,5454,21,Tuesday,Hostel_B,59.546069,43.90,19.00,NaN,1.04,-43.2,NaN,39.062429,88.0,14.41,Low
4,5512,7,Thursday,Lab_Block_B,50.213885,46.81,13.75,31.1,0.62,-48.3,1.70,43.310416,40.0,7.15,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5570,2769,6,Thursday,Admin_Block,20.700667,5.00,6.24,22.8,0.53,-42.9,0.83,4.887468,30.0,9.34,Low
5571,2738,19,Tuesday,Admin_Block,41.242511,50.04,8.45,18.4,1.66,-44.6,2.61,39.449502,45.0,11.62,Low
5572,4241,22,Friday,Lecture_Hall_1,6.874638,5.00,2.14,20.6,0.26,-42.6,0.56,4.883838,5.0,0.00,Low
5573,6306,12,Tuesday,Lecture_Hall_2,161.778438,173.69,65.59,63.5,2.71,-57.5,4.10,148.431633,220.0,64.33,Low


In [8]:
# 2D. Remove duplicates 

df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)                     
df

,Record_ID,Timestamp_Hour,Day_of_Week,Location,Num_Connected_Devices,Bandwidth_Usage_Mbps,AP_Load_Percent,Network_Latency_ms,Packet_Loss_Rate_Percent,Signal_Strength_dBm,Retransmission_Rate_Percent,Throughput_Mbps,Active_Sessions,Channel_Utilization_Percent,Congestion_Level
0,5077,15,Friday,Cafeteria,136.904535,159.14,36.80,39.7,1.72,-55.6,2.82,137.271756,210.0,28.86,Low
1,4532,9,Monday,Library,58.567281,64.59,26.38,34.9,1.22,-47.2,2.06,52.816405,80.0,22.63,Low
2,1498,20,Wednesday,Lecture_Hall_1,5.000000,5.00,2.73,14.9,0.15,-40.7,0.29,4.606722,30.0,5.91,Low
3,5454,21,Tuesday,Hostel_B,59.546069,43.90,19.00,NaN,1.04,-43.2,NaN,39.062429,88.0,14.41,Low
4,5512,7,Thursday,Lab_Block_B,50.213885,46.81,13.75,31.1,0.62,-48.3,1.70,43.310416,40.0,7.15,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5504,2769,6,Thursday,Admin_Block,20.700667,5.00,6.24,22.8,0.53,-42.9,0.83,4.887468,30.0,9.34,Low
5505,2738,19,Tuesday,Admin_Block,41.242511,50.04,8.45,18.4,1.66,-44.6,2.61,39.449502,45.0,11.62,Low
5506,4241,22,Friday,Lecture_Hall_1,6.874638,5.00,2.14,20.6,0.26,-42.6,0.56,4.883838,5.0,0.00,Low
5507,6306,12,Tuesday,Lecture_Hall_2,161.778438,173.69,65.59,63.5,2.71,-57.5,4.10,148.431633,220.0,64.33,Low


In [9]:
# 2E. Fix impossible values (negatives in device count)

df['Num_Connected_Devices'] = df['Num_Connected_Devices'].abs()
df

,Record_ID,Timestamp_Hour,Day_of_Week,Location,Num_Connected_Devices,Bandwidth_Usage_Mbps,AP_Load_Percent,Network_Latency_ms,Packet_Loss_Rate_Percent,Signal_Strength_dBm,Retransmission_Rate_Percent,Throughput_Mbps,Active_Sessions,Channel_Utilization_Percent,Congestion_Level
0,5077,15,Friday,Cafeteria,136.904535,159.14,36.80,39.7,1.72,-55.6,2.82,137.271756,210.0,28.86,Low
1,4532,9,Monday,Library,58.567281,64.59,26.38,34.9,1.22,-47.2,2.06,52.816405,80.0,22.63,Low
2,1498,20,Wednesday,Lecture_Hall_1,5.000000,5.00,2.73,14.9,0.15,-40.7,0.29,4.606722,30.0,5.91,Low
3,5454,21,Tuesday,Hostel_B,59.546069,43.90,19.00,NaN,1.04,-43.2,NaN,39.062429,88.0,14.41,Low
4,5512,7,Thursday,Lab_Block_B,50.213885,46.81,13.75,31.1,0.62,-48.3,1.70,43.310416,40.0,7.15,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5504,2769,6,Thursday,Admin_Block,20.700667,5.00,6.24,22.8,0.53,-42.9,0.83,4.887468,30.0,9.34,Low
5505,2738,19,Tuesday,Admin_Block,41.242511,50.04,8.45,18.4,1.66,-44.6,2.61,39.449502,45.0,11.62,Low
5506,4241,22,Friday,Lecture_Hall_1,6.874638,5.00,2.14,20.6,0.26,-42.6,0.56,4.883838,5.0,0.00,Low
5507,6306,12,Tuesday,Lecture_Hall_2,161.778438,173.69,65.59,63.5,2.71,-57.5,4.10,148.431633,220.0,64.33,Low


In [10]:
# 2F. Handle outliers using IQR method

def cap_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)
    return df

numeric_cols = ['Num_Connected_Devices','Bandwidth_Usage_Mbps','Network_Latency_ms',
                'Packet_Loss_Rate_Percent','AP_Load_Percent','Throughput_Mbps',
                'Active_Sessions','Channel_Utilization_Percent']

for col in numeric_cols:
    df = cap_outliers(df, col)

df

,Record_ID,Timestamp_Hour,Day_of_Week,Location,Num_Connected_Devices,Bandwidth_Usage_Mbps,AP_Load_Percent,Network_Latency_ms,Packet_Loss_Rate_Percent,Signal_Strength_dBm,Retransmission_Rate_Percent,Throughput_Mbps,Active_Sessions,Channel_Utilization_Percent,Congestion_Level
0,5077,15,Friday,Cafeteria,136.904535,159.14,36.80,39.7,1.72,-55.6,2.82,137.271756,210.0,28.86,Low
1,4532,9,Monday,Library,58.567281,64.59,26.38,34.9,1.22,-47.2,2.06,52.816405,80.0,22.63,Low
2,1498,20,Wednesday,Lecture_Hall_1,5.000000,5.00,2.73,14.9,0.15,-40.7,0.29,4.606722,30.0,5.91,Low
3,5454,21,Tuesday,Hostel_B,59.546069,43.90,19.00,NaN,1.04,-43.2,NaN,39.062429,88.0,14.41,Low
4,5512,7,Thursday,Lab_Block_B,50.213885,46.81,13.75,31.1,0.62,-48.3,1.70,43.310416,40.0,7.15,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5504,2769,6,Thursday,Admin_Block,20.700667,5.00,6.24,22.8,0.53,-42.9,0.83,4.887468,30.0,9.34,Low
5505,2738,19,Tuesday,Admin_Block,41.242511,50.04,8.45,18.4,1.66,-44.6,2.61,39.449502,45.0,11.62,Low
5506,4241,22,Friday,Lecture_Hall_1,6.874638,5.00,2.14,20.6,0.26,-42.6,0.56,4.883838,5.0,0.00,Low
5507,6306,12,Tuesday,Lecture_Hall_2,161.778438,173.69,65.59,63.5,2.71,-57.5,4.10,148.431633,220.0,64.33,Low


In [11]:
# 2G. Impute missing values

imputer = KNNImputer(n_neighbors=5)
df[numeric_cols] = imputer.fit_transform(df[numeric_cols])
df

,Record_ID,Timestamp_Hour,Day_of_Week,Location,Num_Connected_Devices,Bandwidth_Usage_Mbps,AP_Load_Percent,Network_Latency_ms,Packet_Loss_Rate_Percent,Signal_Strength_dBm,Retransmission_Rate_Percent,Throughput_Mbps,Active_Sessions,Channel_Utilization_Percent,Congestion_Level
0,5077,15,Friday,Cafeteria,136.904535,159.14,36.80,39.70,1.72,-55.6,2.82,137.271756,210.0,28.86,Low
1,4532,9,Monday,Library,58.567281,64.59,26.38,34.90,1.22,-47.2,2.06,52.816405,80.0,22.63,Low
2,1498,20,Wednesday,Lecture_Hall_1,5.000000,5.00,2.73,14.90,0.15,-40.7,0.29,4.606722,30.0,5.91,Low
3,5454,21,Tuesday,Hostel_B,59.546069,43.90,19.00,31.24,1.04,-43.2,NaN,39.062429,88.0,14.41,Low
4,5512,7,Thursday,Lab_Block_B,50.213885,46.81,13.75,31.10,0.62,-48.3,1.70,43.310416,40.0,7.15,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5504,2769,6,Thursday,Admin_Block,20.700667,5.00,6.24,22.80,0.53,-42.9,0.83,4.887468,30.0,9.34,Low
5505,2738,19,Tuesday,Admin_Block,41.242511,50.04,8.45,18.40,1.66,-44.6,2.61,39.449502,45.0,11.62,Low
5506,4241,22,Friday,Lecture_Hall_1,6.874638,5.00,2.14,20.60,0.26,-42.6,0.56,4.883838,5.0,0.00,Low
5507,6306,12,Tuesday,Lecture_Hall_2,161.778438,173.69,65.59,63.50,2.71,-57.5,4.10,148.431633,220.0,64.33,Low


In [12]:
# this was the first part of Data preprocessing and we handled here all cleaning issues systematically and stored the dataset in new CSV file.
# refer to next jupitor notebook : 02_eda.ipynb .

df.to_csv("Dataset_campus_wifi_Cleaned.csv", index=False)
